In [55]:
import cv2
import numpy as np

# =====================================================
# Load Image
# =====================================================

image = cv2.imread("terlalu-matang_360.jpg")

if image is None:
    print("Error: Image not found!")
    exit()

image = cv2.resize(image, (500, 500))

# Keep original image
original = image.copy()

# Copies for each processing stage
preprocess_img = image.copy()
hsv_display = image.copy()
segmentation_img = image.copy()
contour_img = image.copy()
final_result = image.copy()

# =====================================================
# Step 1 : Image Preprocessing
# =====================================================

# Contrast Enhancement (CLAHE)
lab = cv2.cvtColor(preprocess_img, cv2.COLOR_BGR2LAB)

l, a, b = cv2.split(lab)

clahe = cv2.createCLAHE(clipLimit=2.0,
                        tileGridSize=(8, 8))

l = clahe.apply(l)

lab = cv2.merge((l, a, b))

preprocess_img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

# =====================================================
# Step 2 : RGB -> HSV
# =====================================================

hsv = cv2.cvtColor(preprocess_img, cv2.COLOR_BGR2HSV)

# =====================================================
# Step 3 : Image Segmentation
# =====================================================

lower_green = np.array([35,40,40])
upper_green = np.array([85,255,255])

lower_yellow = np.array([15,40,40])
upper_yellow = np.array([35,255,255])

lower_black = np.array([0, 0, 0])
upper_black = np.array([180, 255, 70])

lower_brown = np.array([5,40,20])
upper_brown = np.array([20,255,220])

mask_brown = cv2.inRange(
    hsv,
    lower_brown,
    upper_brown
)

mask_green = cv2.inRange(hsv,
                         lower_green,
                         upper_green)

mask_yellow = cv2.inRange(hsv,
                          lower_yellow,
                          upper_yellow)

mask_black = cv2.inRange(
    hsv,
    lower_black,
    upper_black
)

mask_brown = cv2.inRange(
    hsv,
    lower_brown,
    upper_brown
)

mask = cv2.bitwise_or(mask_green,
                      mask_yellow)

mask = cv2.bitwise_or(mask, mask_brown)

mask = cv2.bitwise_or(mask,
                      mask_black)

# Show segmented banana
segmentation_img = cv2.bitwise_and(
    preprocess_img,
    preprocess_img,
    mask=mask
)

# =====================================================
# Step 4 : Morphological Operation
# =====================================================

kernel = np.ones((5,5), np.uint8)

mask = cv2.morphologyEx(mask,
                        cv2.MORPH_OPEN,
                        kernel)

mask = cv2.morphologyEx(mask,
                        cv2.MORPH_CLOSE,
                        kernel)

# =====================================================
# Step 5 : Contour Detection
# =====================================================

contours, hierarchy = cv2.findContours(
    mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

if len(contours) > 0:

    largest = max(contours,
                  key=cv2.contourArea)

    cv2.drawContours(
        contour_img,
        [largest],
        -1,
        (0,255,0),
        3
    )

# =====================================================
# Step 6 : Feature Extraction
# =====================================================

green_pixels = cv2.countNonZero(mask_green)

yellow_pixels = cv2.countNonZero(mask_yellow)

black_pixels = cv2.countNonZero(mask_black)

brown_pixels = cv2.countNonZero(mask_brown)

total = green_pixels + yellow_pixels + black_pixels + brown_pixels

if total == 0:
    total = 1

green_ratio = green_pixels / total
yellow_ratio = yellow_pixels / total
black_ratio = black_pixels / total
brown_ratio = brown_pixels / total

print("--------------------------------") 
print("Average HSV") 
print("--------------------------------") 
print("Hue :", H) 
print("Saturation :", S) 
print("Value :", V)
print("--------------------------------") 
print("Colour Ratio")
print("--------------------------------") 
print("Green :", green_ratio)
print("Yellow:", yellow_ratio)
print("Black :", black_ratio)
print("Brown :",brown_ratio)

# =====================================================
# Step 7 : Rule-Based Classification
# =====================================================

if green_ratio >= 0.50:
    label = "mentah"

elif green_ratio >= 0.20:
    label = "setengah-matang"

elif black_ratio >= 0.035 and V < 175:
    label = "terlalu-matang"

else:
    label = "matang"

print("Prediction :", label)

# =====================================================
# Step 8 : Display Final Result
# =====================================================

cv2.putText(
    final_result,
    label,
    (20,40),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (0,0,255),
    2
)

# Draw contour on final image
if len(contours) > 0:

    cv2.drawContours(
        final_result,
        [largest],
        -1,
        (0,255,0),
        3
    )

# =====================================================
# Display All Results
# =====================================================

cv2.imshow("Original Image", original)

cv2.imshow("Preprocessed Image", preprocess_img)

cv2.imshow("HSV Image", hsv)

cv2.imshow("Segmentation", segmentation_img)

cv2.imshow("Mask", mask)

cv2.imshow("Contour Detection", contour_img)

cv2.imshow("Final Result", final_result)

while True:

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q') or key == 27:
        break

cv2.destroyAllWindows()

--------------------------------
Average HSV
--------------------------------
Hue : 18.772515890583435
Saturation : 224.26956115590033
Value : 172.85277095221656
--------------------------------
Colour Ratio
--------------------------------
Green : 0.0
Yellow: 0.2975914134207892
Black : 0.13563472964627213
Brown : 0.5667738569329387
Prediction : terlalu-matang
